# EDB — Export Performance Indicators

Sri Lanka's Export Development Board publishes annual **Export Performance
Indicator** reports: export value by product and destination market, as PDFs.

Connector: `ceynex/data/connectors/edb.py` (owner: M3 Fernando).
Editions configured in `ceynex/data/connectors/apparel_sources.py`.

EDB is the largest single contributor to `fact_trade` in production — and it is
the source that caused the most instructive bug in the project, which §5 of this
notebook reproduces from committed data.

In [1]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

import _common as cx

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Three editions, two different page layouts

| Edition | Layout | Covers | Tables |
|---|---|---|---|
| 2023 | `annual` | 2019–2023 | 17 apparel/textile sub-categories |
| 2024 | `annual` | 2020–2024 | 17 |
| 2009–2018 | `archive` | **2014–2018 only** | 4, much coarser |

Two things to note:

**`latest_year == edition_year` for both annual editions.** Do not assume a
publication lag — pass the year explicitly per edition.

**The archive volume is titled "2009–2018" but its apparel tables only span
2014–2018.** The title describes the bound volume, not the table. Years
2009–2013 are not in that file.

In [2]:
EDB_DIR = cx.CORE / "data" / "raw" / "edb" / "manual"

editions = {
    2023: "export-performance-indicators-of-sri-lanka-2023.pdf",
    2024: "export-performance-indicators-of-sri-lanka-2024.pdf",
    2018: "export-performance-indicators-2009-2018.pdf",
}

pdfs = {}
for year, filename in editions.items():
    path = EDB_DIR / filename
    if path.exists():
        pdfs[year] = path
        print(f"{year}  staged  {filename}  ({path.stat().st_size:,} bytes)")
    else:
        print(f"{year}  MISSING {filename}")

if not pdfs:
    print(f"\nexpected under: {EDB_DIR}")
    print("how to get it:  the EPI reports are free downloads from the EDB")
    print("                e-service books portal. Save each under the exact")
    print("                filename above. The PDFs are gitignored, the paths are not.")

2023  MISSING export-performance-indicators-of-sri-lanka-2023.pdf
2024  MISSING export-performance-indicators-of-sri-lanka-2024.pdf
2018  MISSING export-performance-indicators-2009-2018.pdf

expected under: /ml/CeyNex/ceynex-core/data/raw/edb/manual
how to get it:  the EPI reports are free downloads from the EDB
                e-service books portal. Save each under the exact
                filename above. The PDFs are gitignored, the paths are not.


## 2. Units — a mistake that was caught, and would have been invisible

The tables are headed **"Value in US$ Millions"**, so `to_fact_trade`
multiplies by 1,000,000.

An earlier version of this connector assumed US$ **thousands** and multiplied
by 1,000. Every EDB figure was **1,000× too small**. Nothing crashed and no
test failed — the numbers were simply wrong, and plausibly shaped. It was found
only by reading the real PDF's header text.

This is the case for the team rule that every number gets checked against the
real world once before it goes in a document.

## 3. Two parsing traps

**`pdfplumber.extract_table()` silently corrupts the 2023 edition.** Rows
collapse into three mangled cells, while the 2024 edition parses cleanly with
the same call. The connector therefore **never calls it** — it parses the raw
text layer with `parse_table_page` instead.

**The archive layout has a different number shape.** The annual layout is
5 year-values then one share (7 numbers per row). The archive layout interleaves
a share after *every* year (11 numbers). Matching the wrong pattern returns
**zero rows rather than raising**, which is easy to miss without checking real
output.

## 4. `table_id` is not a product key

This one is worth stating carefully, because it looks like a perfectly good
join key.

> Table **25.89** is *"Made-Up Textile Articles"* in the 2023 edition and
> *"Made-Up Clothing Accessories"* in the 2024 edition. Table 25.94 similarly
> swaps meaning.

EDB renumbers tables between editions. So `table_id` is a **page locator for the
edition it came from**, never a stable product identifier. `to_fact_trade` keys
`source_hash` on the `product` text for exactly this reason.

Joining two editions on `table_id` would merge two different products into one
series without any error.

## 5. The `APPAREL` / `APPREL` bug — reproducible from committed data

EDB is the **only** connector that writes `item` from free text: the
`Product :` line of each table page. Comtrade emits fixed slugs; JAAF a single
constant. So every unnormalised `item` in `fact_trade` came from here.

`item` is column 2 of the 7-column upsert key. A spelling variant is therefore a
**different identity**, and inserts a new row instead of updating the existing
one.

> Sri Lanka's apparel exports to the UAE exist every year from 2014 to 2024.
> But 2014–2018 was filed under `APPAREL` and 2019–2024 under `APPREL` — a typo
> in a later edition. The system saw two unrelated 5-year series instead of one
> 11-year one.
>
> The registered forecasting model was fitted on **5 points** and predicted
> **2019** — a year that had already happened.

The fix is `reference/item_vocabulary.csv`: a controlled vocabulary every
product label must resolve through before it is written. The cells below run
against the committed file, so they work with no PDFs staged.

In [3]:
vocab = cx.reference("item_vocabulary.csv")
print(f"{len(vocab)} entries in item_vocabulary.csv\n")
display(vocab)

20 entries in item_vocabulary.csv



,normalized_label,canonical_item,granularity
0,apparel and textiles,apparel_textiles_edb,total
1,apparel,apparel_edb,group
2,apprel,apparel_edb,group
3,textiles,textiles_edb,group
4,t shirts,t_shirts,component
5,hosiery,hosiery,component
6,mens outerwear,mens_outerwear,component
7,womens outerwear,womens_outerwear,component
8,mens and womens under garments,under_garments,component
9,babies garments,babies_garments,component


In [4]:
import sys as _sys

_sys.path.insert(0, str(cx.CORE))
_sys.path.insert(0, str(cx.WORKSPACE / "ceynex-contracts"))

try:
    from ceynex.data.crosswalk import ItemVocabularyError, canonical_item
except ImportError as exc:
    print("could not import the crosswalk:", exc)
    canonical_item = None

if canonical_item is not None:
    print("The typo and the correct spelling now resolve to the same item:\n")
    for label in ["APPAREL", "APPREL", "Apparel", "APPAREL & TEXTILES", "TEXTILES"]:
        try:
            print(f"  {label!r:24} -> {canonical_item(label)!r}")
        except ItemVocabularyError as exc:
            print(f"  {label!r:24} -> rejected: {exc}")

The typo and the correct spelling now resolve to the same item:

  'APPAREL'                -> 'apparel_edb'
  'APPREL'                 -> 'apparel_edb'
  'Apparel'                -> 'apparel_edb'
  'APPAREL & TEXTILES'     -> 'apparel_textiles_edb'
  'TEXTILES'               -> 'textiles_edb'


In [5]:
if canonical_item is not None:
    print("An unknown label is now a loud error, not a silent new series:\n")
    try:
        canonical_item("APPARELL")
    except ItemVocabularyError as exc:
        print("  ItemVocabularyError:", exc)

An unknown label is now a loud error, not a silent new series:

  ItemVocabularyError: "'APPARELL' (normalized 'apparell') is not in reference/item_vocabulary.csv — add a row rather than letting a new spelling create a second series"


**What the fix produced:** 22 distinct product names collapsed to 18, and
12,256 rows became 12,113 once 143 exact duplicates were removed. The UAE
apparel series became one 11-year series, and the model's next prediction moved
from 2019 to 2025.

> ⚠️ **The `granularity` column above is `PROPOSED`, not verified.** EDB's
> labels describe a three-level tree (total / group / component) and nobody has
> yet checked against a real EPI PDF that the group figures sum to the total.
> Until someone does, the only safe reading is *"component"* vs *"not a
> component"*.

## 6. Load the PDFs

In [6]:
raw = None
if pdfs:
    try:
        from ceynex.data.connectors.edb import parse_pdf_bytes

        frames = []
        for year, path in pdfs.items():
            layout = "archive" if year == 2018 else "annual"
            frames.append(parse_pdf_bytes(path.read_bytes(), year, year, layout))
        raw = pd.concat(frames, ignore_index=True)
        print(f"{len(raw):,} raw rows")
        print("columns:", list(raw.columns))
        display(raw.head())
    except ImportError as exc:
        print("could not import the EDB parser:", exc)
        print("This connector needs Python 3.12 — see the README.")
else:
    print("skipped — no EDB PDFs staged")

skipped — no EDB PDFs staged


## 7. Products and markets

Each row is one market's five-year series for one product, so the year columns
are `year_minus4` … `year_latest` relative to that edition's `latest_year`.

In [7]:
if raw is not None and not raw.empty:
    print("products per edition:")
    print(raw.groupby("edition_year")["product"].nunique().to_string())
    print("\nmarkets per edition:")
    print(raw.groupby("edition_year")["market"].nunique().to_string())
    print("\ndistinct product labels across all editions:")
    for product in sorted(raw["product"].unique()):
        print("  -", product)
else:
    print("skipped — no data")

skipped — no data


## 8. Does every label resolve?

The pipeline refuses to write a label that is not in the vocabulary. Running
that check here shows whether a newly staged edition has introduced a spelling
the vocabulary does not yet know about — which is exactly the condition that
used to split a series in half.

In [8]:
if raw is not None and not raw.empty and canonical_item is not None:
    unresolved = []
    resolved = {}
    for label in sorted(raw["product"].unique()):
        try:
            resolved[label] = canonical_item(label)
        except ItemVocabularyError:
            unresolved.append(label)

    print(f"{len(resolved)} label(s) resolved, {len(unresolved)} unresolved\n")
    for label, item in resolved.items():
        print(f"  {label[:50]:52} -> {item}")
    if unresolved:
        print("\nUNRESOLVED — `make ingest` would raise ItemVocabularyError on these:")
        for label in unresolved:
            print("  -", label)
        print("\nAdd a row to reference/item_vocabulary.csv rather than")
        print("letting a spelling variant start a new series.")
else:
    print("skipped — needs both staged PDFs and the crosswalk")

skipped — needs both staged PDFs and the crosswalk


## 9. Top markets

In [9]:
if raw is not None and not raw.empty:
    latest = raw[raw["edition_year"] == raw["edition_year"].max()]
    top = (
        latest.groupby("market")["year_latest"]
        .sum()
        .sort_values(ascending=False)
        .head(15)
    )
    ax = top.sort_values().plot(
        kind="barh", title=f"Top markets, EDB {int(latest['edition_year'].iloc[0])} edition (USD millions)"
    )
    ax.set_xlabel("USD millions")
    plt.tight_layout()
    plt.show()
    display(top.round(1))
else:
    print("skipped — no data")

skipped — no data
